# EPCLM-001 Formal Execution v0.4

Enable one GPU, Internet, and the existing HF_TOKEN/GITHUB_TOKEN secrets. Run All performs pinned asset recovery → correctness → pilot → formal inference/event measurements → analysis → independently verified HF/GitHub return. No training and no 30M. One separate 7,200-second GPU-worker cap. FP32 only. Missing assets or failed correctness stop the run. See FORMAL_EXECUTION.md for recovery and evidence limits.

In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

SOURCE_SHA = "550861d3c6d45a616c483aff360b1612e7798183"
STAGE = "e1-run"  # Default: all stages. A named e1-* stage retains prerequisite gates.
ROOT = Path("/kaggle/working/epclm001-formal-code-" + SOURCE_SHA[:12])
OUTPUT = Path("/kaggle/working/epclm001-e1-formal-output")
ASSETS = Path("/kaggle/working/epclm001-e1-assets")


def run(*args):
    subprocess.run([str(a) for a in args], check=True)


for label in ("HF_TOKEN", "GITHUB_TOKEN"):
    if not os.environ.get(label):
        from kaggle_secrets import UserSecretsClient

        os.environ[label] = UserSecretsClient().get_secret(label)
print({name + "_available": bool(os.environ.get(name)) for name in ("HF_TOKEN", "GITHUB_TOKEN")})
if not ROOT.exists():
    run("git", "init", "--quiet", ROOT)
    run("git", "-C", ROOT, "remote", "add", "origin", "https://github.com/ArcheLabs/mini-cells.git")
    run("git", "-C", ROOT, "config", "remote.origin.promisor", "true")
    run("git", "-C", ROOT, "config", "remote.origin.partialclonefilter", "blob:none")
    run("git", "-C", ROOT, "sparse-checkout", "init", "--no-cone")
    run(
        "git",
        "-C",
        ROOT,
        "sparse-checkout",
        "set",
        "--no-cone",
        "/pyproject.toml",
        "/src/",
        "/shared/",
        "/research/projects/cellular-language-models/",
        "/research/projects/continual-learning/context/",
    )
run("git", "-C", ROOT, "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_SHA)
run("git", "-C", ROOT, "checkout", "--detach", SOURCE_SHA)
observed = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
assert observed == SOURCE_SHA
requirements = {
    "datasets": "datasets>=2.19",
    "tokenizers": "tokenizers>=0.15",
    "huggingface_hub": "huggingface_hub>=0.36.2,<2",
    "matplotlib": "matplotlib",
}
missing = [
    spec for module, spec in requirements.items() if importlib.util.find_spec(module) is None
]
if missing:
    run(sys.executable, "-m", "pip", "install", "-q", *missing)
run(sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", ROOT)
entry = (
    ROOT
    / "research/projects/cellular-language-models/experiments/epclm-001-incremental-execution/run.py"
)
print(
    {
        "source_sha": SOURCE_SHA,
        "command": STAGE,
        "scope": "FP32 1.9M inference only",
        "training_updates": 0,
        "gpu_worker_budget_seconds": 7200,
    }
)
try:
    run(sys.executable, entry, STAGE, "--output", OUTPUT, "--assets", ASSETS, "--publish-github")
except subprocess.CalledProcessError:
    # Also covers watchdog exit: publish existing partial evidence without more GPU work.
    if (OUTPUT / "RUN_MANIFEST.json").exists():
        subprocess.run(
            [
                sys.executable,
                str(entry),
                "e1-analyze",
                "--output",
                str(OUTPUT),
                "--assets",
                str(ASSETS),
                "--publish-github",
            ],
            check=False,
        )
    raise
finally:
    print("Save Version retains raw reports, figures and profiler traces:", OUTPUT)
    print("Pinned inference weights/validation inputs:", ASSETS)
    print(
        "HF controller restores remote stage boundaries; unresolved reservations remain charged. No training, new Seed or 30M run."
    )
    manifest = OUTPUT / "RUN_MANIFEST.json"
    if manifest.exists():
        value = json.loads(manifest.read_text())
        print(
            {
                k: value.get(k)
                for k in (
                    "status",
                    "run_id",
                    "optimizer_updates",
                    "gpu_worker_seconds_observed",
                    "gpu_worker_seconds_charged",
                )
            }
        )